## FAISS: facebook ai similarity search 


In [1]:
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import faiss,FAISS

/tmp/ipykernel_24690/3610382033.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [2]:
import faiss

# 1. Create a basic CPU index (e.g., L2 flat index with dimension 128)
cpu_index = faiss.IndexFlatL2(128)

# 2. Set up GPU resources
res = faiss.StandardGpuResources()

# 3. Move the index to GPU 0
gpu_index = faiss.index_cpu_to_gpu(res, 0, cpu_index)

# Now you can add vectors and search using gpu_index


In [3]:
import faiss

# Get the count of CUDA-capable devices Faiss detects
gpu_count = faiss.get_num_gpus()
print(f"Number of GPUs available to Faiss: {gpu_count}")

if gpu_count > 0:
    print("✅ Faiss GPU support is enabled and active!")
else:
    print("❌ Faiss is running on CPU only.")


Number of GPUs available to Faiss: 1
✅ Faiss GPU support is enabled and active!


In [4]:
from langchain_community.embeddings import OpenAIEmbeddings
from langchain_text_splitters import CharacterTextSplitter

In [5]:
loader = TextLoader("/storage/nvme/projects/AI-Lab/downloads/datasets/speech.txt")

In [6]:
text = loader.load()

In [7]:
text

[Document(metadata={'source': '/storage/nvme/projects/AI-Lab/downloads/datasets/speech.txt'}, page_content='Democracy is a form of government in which people have the power to choose their leaders through elections. It is based on important values such as **freedom, equality, justice, and respect for human rights**.\n\nIn a democracy, every citizen has the right to express their opinion and participate in the development of the country. People can vote for their representatives and hold them accountable for their actions.\n\nHowever, democracy is not only about voting. It also means respecting the rights and opinions of others, following the law, and fulfilling our responsibilities as citizens.\n\nA strong democracy depends on educated, responsible, and active citizens. When people work together and respect one another, democracy becomes stronger.\n\nIn conclusion, democracy gives people a voice and helps create a fair and peaceful society. We should value and protect it for future gen

In [8]:
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=30)

In [9]:
doc = text_splitter.split_documents(text)

In [10]:
print(doc)

[Document(metadata={'source': '/storage/nvme/projects/AI-Lab/downloads/datasets/speech.txt'}, page_content='Democracy is a form of government in which people have the power to choose their leaders through elections. It is based on important values such as **freedom, equality, justice, and respect for human rights**.\n\nIn a democracy, every citizen has the right to express their opinion and participate in the development of the country. People can vote for their representatives and hold them accountable for their actions.\n\nHowever, democracy is not only about voting. It also means respecting the rights and opinions of others, following the law, and fulfilling our responsibilities as citizens.\n\nA strong democracy depends on educated, responsible, and active citizens. When people work together and respect one another, democracy becomes stronger.\n\nIn conclusion, democracy gives people a voice and helps create a fair and peaceful society. We should value and protect it for future gen

In [11]:
embeddings = OpenAIEmbeddings()
db = FAISS.from_documents(documents=doc, embedding=embeddings)


/tmp/ipykernel_24690/678739214.py:1: LangChainDeprecationWarning: The class `OpenAIEmbeddings` was deprecated in LangChain 0.0.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-openai package and should be used instead. To use it run `pip install -U `langchain-openai` and import as `from `langchain_openai import OpenAIEmbeddings``.
  embeddings = OpenAIEmbeddings()


In [12]:
db

In [13]:
## taking vector db and quering it 

In [14]:
query = "what is the speech about?"

In [15]:
docs = db.similarity_search(query)

In [16]:
docs[0].page_content

'Democracy is a form of government in which people have the power to choose their leaders through elections. It is based on important values such as **freedom, equality, justice, and respect for human rights**.\n\nIn a democracy, every citizen has the right to express their opinion and participate in the development of the country. People can vote for their representatives and hold them accountable for their actions.\n\nHowever, democracy is not only about voting. It also means respecting the rights and opinions of others, following the law, and fulfilling our responsibilities as citizens.\n\nA strong democracy depends on educated, responsible, and active citizens. When people work together and respect one another, democracy becomes stronger.\n\nIn conclusion, democracy gives people a voice and helps create a fair and peaceful society. We should value and protect it for future generations.'

## now RAG

In [17]:
retriver = db.similarity_search

In [18]:
docs = retriver(query)
docs

[Document(id='8c43b283-b29d-464f-b545-214dfa597c1f', metadata={'source': '/storage/nvme/projects/AI-Lab/downloads/datasets/speech.txt'}, page_content='Democracy is a form of government in which people have the power to choose their leaders through elections. It is based on important values such as **freedom, equality, justice, and respect for human rights**.\n\nIn a democracy, every citizen has the right to express their opinion and participate in the development of the country. People can vote for their representatives and hold them accountable for their actions.\n\nHowever, democracy is not only about voting. It also means respecting the rights and opinions of others, following the law, and fulfilling our responsibilities as citizens.\n\nA strong democracy depends on educated, responsible, and active citizens. When people work together and respect one another, democracy becomes stronger.\n\nIn conclusion, democracy gives people a voice and helps create a fair and peaceful society. We

# FAISS specific similarity score 

In [19]:
doc_and_dcore = db.similarity_search_with_score(query)

In [20]:
doc_and_dcore

[(Document(id='8c43b283-b29d-464f-b545-214dfa597c1f', metadata={'source': '/storage/nvme/projects/AI-Lab/downloads/datasets/speech.txt'}, page_content='Democracy is a form of government in which people have the power to choose their leaders through elections. It is based on important values such as **freedom, equality, justice, and respect for human rights**.\n\nIn a democracy, every citizen has the right to express their opinion and participate in the development of the country. People can vote for their representatives and hold them accountable for their actions.\n\nHowever, democracy is not only about voting. It also means respecting the rights and opinions of others, following the law, and fulfilling our responsibilities as citizens.\n\nA strong democracy depends on educated, responsible, and active citizens. When people work together and respect one another, democracy becomes stronger.\n\nIn conclusion, democracy gives people a voice and helps create a fair and peaceful society. W

In [21]:
docs_and_scores = db.similarity_search_with_relevance_scores(query)

for doc, relevance in docs_and_scores:
    print("Relevance:", relevance)

Relevance: 0.6433973


In [22]:
results = db.similarity_search_with_relevance_scores(query)

for doc, relevance in results:
    print(relevance)

0.6433973
